In [0]:
# ================================================================
# NOTEBOOK: nb_silver_orderitems_cdc_merge
# PURPOSE:  ADF CDC Parquet → Silver Merge for OrderItems
#
# SOURCE:   bronze/orderitems/   (Parquet)
# TARGET:   silver/orderitems/   (Delta)
#
# HANDLES:
#   INSERT → New row in Silver
#   UPDATE → Existing row updated in Silver
#

# ================================================================


from pyspark.sql import functions as F
from pyspark.sql.functions import col, to_timestamp, upper, trim
from pyspark.sql.window import Window
from delta.tables import DeltaTable


# ================================================================
# PATHS
# ================================================================

CDC_PATH = (
    "abfss://source@stshopsensedevhj.dfs.core.windows.net/"
    "bronze/orderitems/"
)

SILVER_PATH = (
    "abfss://source@stshopsensedevhj.dfs.core.windows.net/"
    "silver/orderitems/"
)


# ================================================================
# STEP 1: READ BRONZE ORDERITEM CHANGES
# ================================================================

cdc_raw = spark.read.parquet(CDC_PATH)

total_rows = cdc_raw.count()

if total_rows == 0:
    print("[INFO] No OrderItems records found.")
    dbutils.notebook.exit("NO_CHANGES")

print(f"[BRONZE] Rows read: {total_rows}")


# ================================================================
# STEP 2: KEEP LATEST ROW PER OrderItemID
# ================================================================
# The Bronze folder can contain multiple versions of the same item.
# We keep the row with the newest LastModifiedDate.
# ================================================================

latest_window = (
    Window
    .partitionBy("OrderItemID")
    .orderBy(
        F.desc("LastModifiedDate")
    )
)

cdc_latest = (
    cdc_raw
    .withColumn(
        "_rn",
        F.row_number().over(latest_window)
    )
    .filter(col("_rn") == 1)
    .drop("_rn")
)

latest_count = cdc_latest.count()

print(f"[LATEST] Latest OrderItemID rows: {latest_count}")


# ================================================================
# STEP 3: TRANSFORMATION FUNCTION
# ================================================================

def transform_orderitems(df):

    return (
        df

        # --------------------------------------------------------
        # Type conversions
        # --------------------------------------------------------

        .withColumn(
            "LastModifiedDate",
            to_timestamp(col("LastModifiedDate"))
        )

        .withColumn(
            "Quantity",
            col("Quantity").cast("integer")
        )

        .withColumn(
            "UnitPrice",
            col("UnitPrice").cast("decimal(10,2)")
        )

        .withColumn(
            "DiscountAmount",
            col("DiscountAmount").cast("decimal(10,2)")
        )

        .withColumn(
            "TotalPrice",
            col("TotalPrice").cast("decimal(10,2)")
        )


        # --------------------------------------------------------
        # Text standardisation
        # --------------------------------------------------------

        .withColumn(
            "Category",
            upper(trim(col("Category")))
        )


        # --------------------------------------------------------
        # Boolean conversion
        # --------------------------------------------------------

        .withColumn(
            "IsGift",
            upper(
                trim(col("IsGift").cast("string"))
            ) == "TRUE"
        )


        # --------------------------------------------------------
        # NetPrice
        #
        # Total item value after discount:
        # UnitPrice × Quantity - DiscountAmount
        # --------------------------------------------------------

        .withColumn(
            "NetPrice",
            F.round(
                (col("UnitPrice") * col("Quantity"))
                - col("DiscountAmount"),
                2
            ).cast("decimal(10,2)")
        )


        # --------------------------------------------------------
        # EffectiveUnitPrice
        #
        # Actual price paid per item after discount.
        #
        # Example:
        # NetPrice = 1800
        # Quantity = 2
        # EffectiveUnitPrice = 900
        # --------------------------------------------------------

        .withColumn(
            "EffectiveUnitPrice",
            F.when(
                col("Quantity") > 0,
                F.round(
                    col("NetPrice") / col("Quantity"),
                    2
                )
            )
            .otherwise(F.lit(None))
            .cast("decimal(10,2)")
        )


        # --------------------------------------------------------
        # DiscountPct
        #
        # DiscountAmount / original total value × 100
        # --------------------------------------------------------

        .withColumn(
            "DiscountPct",
            F.when(
                (
                    col("UnitPrice") * col("Quantity")
                ) > 0,

                F.round(
                    col("DiscountAmount")
                    / (
                        col("UnitPrice") * col("Quantity")
                    )
                    * 100,
                    2
                )
            )
            .otherwise(F.lit(0.0))
            .cast("decimal(10,2)")
        )


        # --------------------------------------------------------
        # IsDiscounted
        # --------------------------------------------------------

        .withColumn(
            "IsDiscounted",
            col("DiscountAmount") > 0
        )


        # --------------------------------------------------------
        # Silver metadata
        # --------------------------------------------------------

        .withColumn(
            "_silver_load_ts",
            F.current_timestamp()
        )

       .withColumn(
           "source",
           F.lit("adf_cdc_parquet")
           )

        .withColumn(
            "_is_deleted",
            F.lit(False)
        )
    )


# Apply transformations
transformed_orderitems = transform_orderitems(cdc_latest)


# ================================================================
# STEP 4: CHECK SILVER DELTA TABLE
# ================================================================

if not DeltaTable.isDeltaTable(spark, SILVER_PATH):
    raise Exception(
        "Silver OrderItems Delta table does not exist. "
        "Run the initial Silver OrderItems notebook first."
    )

silver_df = (
    spark.read
    .format("delta")
    .load(SILVER_PATH)
)

silver = DeltaTable.forPath(
    spark,
    SILVER_PATH
)


# ================================================================
# STEP 5: CHECK SOURCE AND TARGET COLUMNS
# ================================================================

source_columns = set(transformed_orderitems.columns)
target_columns = set(silver_df.columns)

missing_in_source = sorted(
    target_columns - source_columns
)

extra_in_source = sorted(
    source_columns - target_columns
)

print("[SCHEMA CHECK] Missing in transformed source:")
print(missing_in_source)

print("[SCHEMA CHECK] Extra in transformed source:")
print(extra_in_source)


# Stop before merge if Silver expects another missing column
if len(missing_in_source) > 0:
    raise Exception(
        "Merge stopped because these Silver columns are missing "
        f"from transformed OrderItems: {missing_in_source}"
    )


# Select target columns in the same order as Silver
transformed_orderitems = transformed_orderitems.select(
    *silver_df.columns
)


# ================================================================
# STEP 6: MERGE INSERTS AND UPDATES
# ================================================================

(
    silver.alias("s")
    .merge(
        transformed_orderitems.alias("c"),
        "s.OrderItemID = c.OrderItemID"
    )

    # Existing OrderItemID → update
    .whenMatchedUpdateAll()

    # New OrderItemID → insert
    .whenNotMatchedInsertAll()

    .execute()
)

print(
    f"[MERGE] {latest_count} OrderItems merged into Silver."
)


# ================================================================
# STEP 7: FINAL VERIFICATION
# ================================================================

final_df = (
    spark.read
    .format("delta")
    .load(SILVER_PATH)
)

total_count = final_df.count()

active_count = (
    final_df
    .filter(col("_is_deleted") == False)
    .count()
)

deleted_count = (
    final_df
    .filter(col("_is_deleted") == True)
    .count()
)

print(
    f"[DONE] Silver OrderItems:"
    f"\nTotal        : {total_count}"
    f"\nActive       : {active_count}"
    f"\nSoft-deleted : {deleted_count}"
)

[BRONZE] Rows read: 5947
[LATEST] Latest OrderItemID rows: 5945
[SCHEMA CHECK] Missing in transformed source:
[]
[SCHEMA CHECK] Extra in transformed source:
[]
[MERGE] 5945 OrderItems merged into Silver.
[DONE] Silver OrderItems:
Total        : 5945
Active       : 5945
Soft-deleted : 0


In [0]:
display(
    final_df
    .filter(
        col("OrderItemID").isin(
            "ITEM_CDC_105",
            "ITEM_CDC_106",
            "ITEM_CDC_107",
            "ITEM_CDC_108",
            "ITEM_CDC_109",
            "ITEM_CDC_110",
            "ITEM_CDC_111",
            "ITEM_CDC_112"
        )
    )
    .select(
        "OrderItemID",
        "OrderID",
        "Quantity",
        "TotalPrice",
        "LastModifiedDate",
        "_is_deleted"
    )
    .orderBy("OrderItemID")
)

OrderItemID,OrderID,Quantity,TotalPrice,LastModifiedDate,_is_deleted
ITEM_CDC_105,ORD_CDC_111,3,3398.50,2026-07-15T07:41:36.09Z,false
ITEM_CDC_106,ORD_CDC_111,2,598.00,2026-07-15T07:41:36.09Z,false
ITEM_CDC_107,ORD_CDC_112,4,7498.68,2026-07-15T07:41:36.09Z,false
ITEM_CDC_108,ORD_CDC_112,2,999.98,2026-07-15T07:41:36.09Z,false
ITEM_CDC_109,ORD_CDC_111,2,2199.00,2026-07-15T07:48:22.823333Z,false
ITEM_CDC_110,ORD_CDC_111,1,299.00,2026-07-15T07:48:22.823333Z,false
ITEM_CDC_111,ORD_CDC_112,3,5499.01,2026-07-15T07:48:22.823333Z,false
ITEM_CDC_112,ORD_CDC_112,1,499.99,2026-07-15T07:48:22.823333Z,false
